In [ ]:
from pathlib import Path
import sys
import json
from datetime import datetime
from json import JSONDecodeError

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Llm.llm_loader import LLM
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(project_root / '.env')

In [ ]:
def load_data(project_root, dataset_name):
    datasets = [
        (
            'two_table',
            project_root / 'Data' / dataset_name / 'Synthesized_two_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset_name / 'Synthesized_two_table_with_spider_db_id.json')
        ),
        (
            'three_table',
            project_root / 'Data' / dataset_name / 'Synthesized_three_table_with_spider_db_id.json',
            pd.read_json(project_root / 'Data' / dataset_name / 'Synthesized_three_table_with_spider_db_id.json')
        ),
    ]
    return datasets

In [ ]:
def append_log_entry(log_records, dataset_name, data_split, row, tables_from_llm, columns_from_llm,Answer_llm,log_path):
    log_records.append(
        {
            'model': Answer_llm.model_name,
            'provider': Answer_llm.provider,
            'id': f"{dataset_name}-{data_split}-{row['id_']}",
            'spider_db_id': row['Spider_db_id'],
            'question': row['Question'],
            'tables_from_llm': tables_from_llm,
            'columns_from_llm': columns_from_llm
        }
    )
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')

In [ ]:
def run_(project_root,prompt_paths, log_path, datasets,dataset_name,Answer_llm):
    extract_table_prompt_template = prompt_paths['table'].read_text(encoding='utf-8').strip()
    extract_column_prompt_template = prompt_paths['column'].read_text(encoding='utf-8').strip()
    log_records = []
    log_path.write_text(json.dumps(log_records, ensure_ascii=False, indent=2), encoding='utf-8')
    for data_split, data_source, df in datasets:
        for _, row in tqdm(df.iterrows(), total=len(df), desc=data_split):
            schema_path = f"{project_root}/Data/MMQA/schema_descriptions/{row['Spider_db_id']}.csv"
            total_schema_df = pd.read_csv(schema_path)
            extract_table_prompt = (
                extract_table_prompt_template
                .replace('{DATABASE_SCHEMA}', total_schema_df.to_markdown())
                .replace('{QUESTION}', row['Question'])
                .replace('{HINT}', 'No hint')
            )
            tables_from_llm = Answer_llm.query(extract_table_prompt)
            try:
                table_dict = json.loads(tables_from_llm.strip())
                if table_dict:
                    relevant_table_list = table_dict["relevant_tables"]
                else:
                    tables_from_llm = "Empty Resulst:\n" + tables_from_llm.strip()
            except JSONDecodeError:
                relevant_table_list = []
                tables_from_llm = "Invalid JSON:\n" + tables_from_llm.strip()
            except KeyError:
                relevant_table_list = []
                tables_from_llm = "Invalid Key:\n" + tables_from_llm.strip()

            if relevant_table_list:
                relevant_schema_df = total_schema_df[total_schema_df['table_name'].isin(relevant_table_list)]
            else:
                relevant_schema_df = total_schema_df

            extract_column_prompt = (
                extract_column_prompt_template
                .replace('{DATABASE_SCHEMA}', relevant_schema_df.to_markdown())
                .replace('{QUESTION}', row['Question'])
                .replace('{HINT}', 'No hint')
            )
            columns_from_llm = Answer_llm.query(extract_column_prompt)
            append_log_entry(log_records, dataset_name, data_split, row, tables_from_llm, columns_from_llm,Answer_llm,log_path)

    print(f'All responses have been saved to {log_path}')




In [ ]:
dataset_name = 'MMQA'
datasets = load_data(project_root, dataset_name)

ollama_llm_lists = ['gemma3:27b']
gpt_llm_lists = ['gpt-3.5-turbo', 'gpt-4o-mini','gpt-5-mini']
# gpt_llm_lists = ['gpt-3.5-turbo', 'gpt-4o-mini','gpt-5-mini'] 
# gpt-3.5-turbo-0125, gpt-4o-mini-2024-07-18, gpt-5-mini-2025-08-07
# Intelligence: 1, 2, 3
# Speed: 2, 4, 4
# Reasoning tokens: no, no, yes
# Knowledge Cutoff: Sep 01, 2021; Oct 01, 2023; May 31, 2024

In [ ]:
method_ = 'zero_shot'
prompt_paths = {
    'table' : project_root / 'Templates' / method_ / 'extract_relevant_tables.txt',
    'column' : project_root / 'Templates' / method_ / 'extract_relevant_columns.txt'
}

In [ ]:
for model_name in ollama_llm_lists:
    Answer_llm = LLM(model_name = model_name, provider= 'ollama')
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    logs_dir = project_root / 'Logs' / model_name
    logs_dir.mkdir(exist_ok=True)
    log_path = logs_dir / f'{method_}_table2column_{dataset_name}_{run_id}.json'
    run_(project_root,prompt_paths, log_path, datasets, dataset_name, Answer_llm)

In [ ]:
for model_name in gpt_llm_lists:
    Answer_llm = LLM(model_name = model_name, provider= 'openai')
    run_id = datetime.now().strftime('%Y%m%d_%H%M%S')
    logs_dir = project_root / 'Logs' / model_name
    logs_dir.mkdir(exist_ok=True)
    log_path = logs_dir / f'{method_}_table2column_{dataset_name}_{run_id}.json'
    run_(project_root,prompt_paths, log_path, datasets, dataset_name, Answer_llm)